# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [4]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [6]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [8]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context")) #
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [10]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times among the projects listed.'

In [11]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there is a use case related to security. Specifically, the project titled "CreateFlow 41" falls under the domain of Customer Support / Helpdesk, and its description mentions it as "A federated learning toolkit improving privacy in healthcare applications." Additionally, another project, "Pathfinder 24," is in the Healthcare / MedTech domain with a secondary domain of Security, focusing on an AI-powered platform that optimizes logistics routes for sustainability, which could also relate to security considerations.'

In [12]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had generally positive comments about the fintech projects. They described some projects as "a clever solution with measurable environmental benefit," "technically ambitious and well-executed," and "promising idea with robust experimental validation." Additionally, others were praised for "solid work with impressive real-world impact," "excellent code quality and use of open-source libraries," and "conceptually strong but results need more benchmarking." Overall, the judges recognized the technical quality, innovation, and potential real-world impact of the fintech-related projects, with some noting minor areas for improvement.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [13]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [14]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [15]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided data, the project domains mentioned include 'Productivity Assistants', 'E‑commerce / Marketplaces', 'Healthcare / MedTech', and 'Finance / FinTech'. Since this is a small sample, I cannot definitively determine the most common project domain overall. However, among these examples, 'Finance / FinTech' appears twice, suggesting it may be a common domain in this dataset. \n\nIf you need a precise answer based on the full dataset, I would recommend analyzing all entries to see which domain appears most frequently."

In [16]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there is a use case related to security. The project "SecureNest" falls under the domains of E‑commerce / Marketplaces and Legal / Compliance, and involves a document summarization and retrieval system for enterprise knowledge bases, which relates to security and compliance concerns.'

In [17]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges described the fintech project as being conceptually strong, but noted that the results need more benchmarking.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer

`Example query`: “Quarterly revenue of Apple in Q2 2023”

`Justification`:
BM25 directly matches exact tokens like “Apple,” “Q2,” and “2023”, making it ideal for retrieving factual financial documents.
Embeddings might blur these with semantically related but irrelevant texts (e.g., “Apple’s stock growth in 2023”).


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [18]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [19]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [20]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Security," as it is listed as the project domain for one of the projects. However, since the data snippet is limited and only shows a few entries, it is not definitive. \n\nIf considering just this data, "Security" is the most prominent. For a comprehensive answer, a full analysis of all projects in the dataset would be required. \n\nWould you like me to assist further with analyzing the entire dataset?'

In [21]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases explicitly related to security. The highlighted projects focus on privacy improvements in healthcare applications, but there is no mention of security-specific use cases.'

In [22]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had positive comments about the fintech project, Pathfinder 27. They praised its excellent code quality and the use of open-source libraries.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [23]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [24]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [25]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided data, the most common project domain appears to be "Writing & Content," which is mentioned multiple times in the dataset.'

In [26]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security in the provided context. One notable example is "SecureNest," which is described as "A document summarization and retrieval system for enterprise knowledge bases." Additionally, "EchoLens" is mentioned as "A hardware-aware model quantization benchmark suite," which may have security implications related to hardware and data privacy.'

In [27]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had generally positive comments about the fintech projects. For example, the project "Pathfinder 27" received praise for "excellent code quality and use of open-source libraries," and "SecureNest 28" was described as "conceptually strong but results need more benchmarking." Overall, judges highlighted the technical strength, potential for real-world impact, and solid execution of some projects within the fintech domain.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

By creating several embeddings for rephrasings (reformulations) of a query, each version captures a slightly different way of expressing the same intent, allowing the system to retrieve more relevant documents that a single embedding might overlook.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [28]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [29]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [30]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [31]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [32]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [33]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data appears to be "Healthcare / MedTech," as it is mentioned multiple times among the projects listed.'

In [34]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases related to security mentioned. The projects focus on federated learning to improve privacy in healthcare applications, which is related to privacy protection rather than general security.'

In [35]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Based on the provided context, the judges had positive comments about the fintech projects. Specifically, they mentioned that one project was "technically ambitious and well-executed," another was praised for being "solid work with impressive real-world impact," and a third was described as "comprehensive and technically mature." Overall, the judges appreciated the technical quality and potential impact of these fintech-related projects.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [36]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list) #list repetition

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [37]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [38]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain among the listed projects is "Writing & Content," which appears twice in the provided data. Other domains like "Healthcare / MedTech" and "Finance / FinTech" also appear, but less frequently.'

In [39]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there is at least one use case related to security. The project titled "SecureNest 49" focuses on a document summarization and retrieval system for enterprise knowledge bases, which is related to security in the context of enterprise data management and compliance.'

In [40]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges had generally positive comments about the fintech projects. For example, they described the project "SynthMind" as conceptually strong but noted that results need more benchmarking. "PulseAI" was praised as technically ambitious and well-executed. The scores for these projects were also high, with SynthMind receiving a judge score of 9.6 and PulseAI receiving a score of 8.0, indicating favorable evaluations.'

In [41]:
ensemble_retrieval_chain.invoke({"question" : "Which was the highest scored project? Which is about and why it socred so high?"})["response"].content

'The highest scored project is "AutoVerse," with a score of 96. It is an interactive 3D environment designed for generative design and architecture within the Healthcare / MedTech domain and Secondary Domain of QA / Testing / Validation. \n\n"AutoVerse" scored highly because it was recognized as a great innovation, even though it needed stronger evaluation metrics. The high score reflects the judges\' appreciation for its innovative approach and potential impact in its field.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain in the provided data is "Legal / Compliance," which appears twice. Other domains such as "Developer Tools / DevEx," "Customer Support / Helpdesk," and "Writing & Content" also appear twice each, but no domain appears more frequently than "Legal / Compliance."'

In [48]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Yes, there are use cases related to security. Specifically, there are projects such as "MediMind 17" which is described as a medical imaging solution improving early diagnosis, and "SecureNest 12," which is a low-latency inference system for multimodal agents in autonomous systems. Both are categorized under the domain of security.'

In [49]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had a generally positive view of the fintech projects. For example, the project "WealthifyAI 16" was described as having a comprehensive and technically mature approach, with a judge score of 8.4. Similarly, "AutoMate 5" received positive feedback as a forward-looking idea with solid supporting data, and a high judge score of 8.6. Overall, the judges highlighted the projects\' technical ambition, maturity, and potential for impact.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

With short, repetitive FAQs, semantic chunking tends to over-merge because everything looks semantically similar.
To fix it we can use tighten similarity thresholds, or  set fixed-size rules (split first with RecursiveCharacterTextSplitter, then use semantic) to preserve meaningful, consistent chunk boundaries.

Examples:

1. Increasing the threshold amount:
```python
chunker = SemanticChunker(
    embedding_model=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=85  # higher = fewer merges
)
```
2. Set fixed-size rules
```python 
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.experimental.text_splitter import SemanticChunker

# Step 1: Split by size first
char_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
base_chunks = char_splitter.split_text(long_text)

# Step 2: Then apply semantic chunking within each segment
semantic_splitter = SemanticChunker(embedding_model=embeddings)
final_chunks = [semantic_splitter.create_documents([chunk]) for chunk in base_chunks]
```

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

#### Synthetic Dataset Generation

In [50]:
# Enable LangSmith
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"]= "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Introduce your LangSmith API key: ")

In [51]:
# Setting LangSmith project
os.environ["LANGCHAIN_PROJECT"] = "Retrieval Evaluation"

In [52]:

# Loading data
from langchain_community.document_loaders.csv_loader import CSVLoader

ragas_loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

ragas_usecase_data = ragas_loader.load()




In [53]:
ragas_usecase_data[0].metadata

{'source': './data/Projects_with_Domains.csv',
 'row': 0,
 'Project Title': 'InsightAI 1',
 'Project Domain': 'Security',
 'Secondary Domain': 'Finance / FinTech',
 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.',
 'Judge Comments': 'Technically ambitious and well-executed.',
 'Score': '85',
 'Project Name': 'Project Aurora',
 'Judge Score': '9.5'}

Since Ragas' knowledge graph generation requires a minimum of 100 tokens per document, we artificially expand the dataset by enriching each document's page_content with additional metadata and descriptive text.

In [54]:
# Expanded way to add more content to page_content
for doc in ragas_usecase_data:
    # Extract all metadata
    title = doc.metadata.get("Project Title", "")
    domain = doc.metadata.get("Project Domain", "")
    secondary_domain = doc.metadata.get("Secondary Domain", "")
    description = doc.metadata.get("Description", "")
    judge_comments = doc.metadata.get("Judge Comments", "")
    score = doc.metadata.get("Score", "")
    project_name = doc.metadata.get("Project Name", "")
    judge_score = doc.metadata.get("Judge Score", "")
    
    # Create expanded content with more context
    expanded_content = f"""
    {doc.page_content}
    
    This project represents a significant contribution to the {domain} domain. 
    The project titled "{title}" focuses on {description.lower()}. 
    With a score of {score} and judge score of {judge_score}, this project demonstrates 
    strong technical merit and innovation. Judge feedback indicates: {judge_comments}. 
    The project's success in the {domain} domain, with secondary focus on {secondary_domain}, 
    shows its potential for real-world application and impact.
    
    The project's technical approach and implementation demonstrate advanced capabilities 
    in the {domain} field. The combination of technical excellence and practical applicability 
    makes this project noteworthy. The project's success is evidenced by its high score 
    and positive judge feedback, indicating strong potential for future development.
    
    """
    
    # Update the document
    doc.page_content = expanded_content

In [55]:
# Number of documents to add to the knowledge graph
len(ragas_usecase_data)

50

In [56]:
# Verify the content of our list of documents
ragas_usecase_data

[Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='\n    \n\n    This project represents a significant contribution to the Security domain. \n    The project titled "InsightAI 1" focuses on a low-latency inference system for multimodal agents in autonomous systems.. \n    With a score of 85 and judge score of 9.5, this project demonstrates \n    strong technical merit and innovation. Judge feedback indicates: Technically ambitious and well-executed.. \n    The project\'s success in the Security domain, with secondary focus on Finance / FinTech, \n    shows its potential for real-world application and impact.\n\n    The pro

In [57]:
# Verify the length to properly use RAGAS
len(ragas_usecase_data[0].page_content)

923

Since we artificially expanded the dataset for Ragas SDG, we need to create new vectorstores using the expanded dataset and then recreate all retrievers (naive, BM25, compression, multi-query, parent document, ensemble, semantic).

In [58]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore_ragas = Qdrant.from_documents(
    ragas_usecase_data,
    embeddings,
    location=":memory:",
    collection_name = 'Ragas_Expanded_Usecases'
)

In [59]:
# Naive retriever------------------------------------------------------
naive_retriever = vectorstore_ragas.as_retriever(search_kwargs={"k":10})

# BM25 retriever-------------------------------------------------
bm25_retriever = BM25Retriever.from_documents(ragas_usecase_data)

# Contextual Compression --------------------
compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor = compressor, base_retriever=naive_retriever
)

# Multi-query retriever-----------------------------
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever = naive_retriever,
    llm = chat_model
)

# Parent Document retriever-----
parent_docs = ragas_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

# Vector store for parent documents
client  = QdrantClient(location = ":memory:")
client.create_collection(
    collection_name = 'ragas_full_documents',
    vectors_config = models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name='ragas_full_documents',
    embedding= OpenAIEmbeddings(model='text-embedding-3-small'),
    client= client
)

store= InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore = store,
    child_splitter = child_splitter
)

parent_document_retriever.add_documents(parent_docs, ids=None)

# Ensemble retriever-------------------------------------------------------------------------------------------------------
retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list) #list repetition

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

# Semantic retriever---------------
semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type = "percentile"
)

semantic_documents = semantic_chunker.split_documents(ragas_usecase_data)

semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Ragas_Usecase_Data_Semantic_Chunks"
)

semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})


Now we can continue with our Synthetic Data Generation!

In [60]:
# Create generators for testset generator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm =ChatOpenAI(model='gpt-4.1-nano')

generator_llm=LangchainLLMWrapper(llm)
generator_embeddings=LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))





C:\Users\Inés\AppData\Local\Temp\ipykernel_84912\339441969.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm=LangchainLLMWrapper(llm)
C:\Users\Inés\AppData\Local\Temp\ipykernel_84912\339441969.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings=LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))


In [61]:
# Knowledge graph
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

In [62]:
# Add document to knowledge graph
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in ragas_usecase_data:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 50, relationships: 0)

In [63]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=ragas_usecase_data, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying SummaryExtractor:   0%|          | 0/50 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/50 [00:00<?, ?it/s]

Applying EmbeddingExtractor:   0%|          | 0/50 [00:00<?, ?it/s]

Applying ThemesExtractor:   0%|          | 0/50 [00:00<?, ?it/s]

Applying NERExtractor:   0%|          | 0/50 [00:00<?, ?it/s]

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 50, relationships: 1332)

We are going to create some personas about how I imagine users could ask this questions... ragas also offers automatic persona generation. But for the shake of the exercise, let's do it manually!

In [64]:
# Personas
from ragas.testset.persona import Persona

persona_new_joinee = Persona(
    name="New Joinee",
    role_description=(
        "Needs fast inspiration and clear starters. Asks for recurrent domain and secondary from past projects,"
        "with one-sentence summaries and 1–2 concrete next steps. "
        "Prefers examples that are beginner-friendly and feasible within the bootcamp timeline."
    ),
)

persona_high_achiever = Persona(
    name="High Achiever",
    role_description=(
        "Wants the highest-scoring projects and the reasons they won. Requests ranked results with "
        "Score/Judge Score thresholds and brief evidence pulled from judge comments. Seeks patterns "
        "to replicate (common techniques, domains, deliverables) to maximize grading outcomes."
    ),
)

persona_innovator = Persona(
    name="Innovator",
    role_description=(
        "Aims for a job-aligned, startup-style idea that stays feasible. Looks for cross-domain "
        "combinations from past projects, highlights of novel methods in descriptions, and why "
        "judges valued them. Prefers suggestions that balance originality with a realistic 4–6 week scope."
    ),
)

personas = [persona_new_joinee, persona_high_achiever, persona_innovator]


For this CSV-based project dataset, we are using only *SingleHopSpecificQuerySynthesizer* with a weight of 1. I have considered the optimal approach since Multi-hop synthesizers like MultiHopAbstractQuerySynthesizer and MultiHopSpecificQuerySynthesizer are better suited for large knowledge bases with complex relationships that require complex reasoning.



In [65]:
# Synthesizers
from ragas.testset.synthesizers import SingleHopSpecificQuerySynthesizer
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1),
    
]

In [66]:
from ragas.testset import TestsetGenerator
testset_generator = TestsetGenerator(knowledge_graph = kg, llm= generator_llm, embedding_model = generator_embeddings, persona_list= personas)
testset = testset_generator.generate(testset_size=10, query_distribution= query_distribution)

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

In [67]:
testset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How does the InsightAI 1 project demonstrate i...,[\n \n\n This project represents a signi...,The InsightAI 1 project demonstrates innovatio...,single_hop_specific_query_synthesizer
1,What is DevEx in dev tools?,[\n \n\n This project represents a signi...,The project demonstrates strong technical meri...,single_hop_specific_query_synthesizer
2,WealthifyAI 3 good?,[\n \n\n This project represents a signi...,"The project titled ""WealthifyAI 3"" focuses on ...",single_hop_specific_query_synthesizer
3,Finance FinTech project good?,[\n \n\n This project represents a signi...,"The project, titled ""MediMind 4,"" focuses on a...",single_hop_specific_query_synthesizer
4,Considering the project's focus on the Healthc...,[\n \n\n This project represents a signi...,"The project ""AutoMate 5"" shows its potential f...",single_hop_specific_query_synthesizer
5,Can you provide a brief overview of TrendLens ...,[\n \n\n This project represents a signi...,TrendLens 6 is an interactive 3D environment f...,single_hop_specific_query_synthesizer
6,PlanPilot 7 what it do for real world?,[\n \n\n This project represents a signi...,PlanPilot 7 is a project that focuses on a low...,single_hop_specific_query_synthesizer
7,Why was the InsightAI 8 project considered suc...,[\n \n\n This project represents a signi...,The project's success in the Legal / Complianc...,single_hop_specific_query_synthesizer
8,Whaat is ChatBridge 9 and how does it contribu...,[\n \n\n This project represents a signi...,ChatBridge 9 is a bioinformatics pipeline leve...,single_hop_specific_query_synthesizer
9,Can you provide a brief overview of MediMind 1...,[\n \n\n This project represents a signi...,MediMind 10 is a real-time object tracking sys...,single_hop_specific_query_synthesizer


Try the Abstract method!

In [69]:
# testset.to_pandas()

#### Evaluation

We are going to implement the evaluation using *retriver specific* Ragas metrics (context precision, context recall, and faithfulness) integrated with LangSmith through **EvaluatorChain wrappers**. 

In [70]:
# Define all retrievers to evaluate
retrievers_to_evaluate = {
    "naive": naive_retriever,
    "bm25": bm25_retriever,
    "compression": compression_retriever,
    "multi_query": multi_query_retriever,
    "parent_document": parent_document_retriever,
    "ensemble": ensemble_retriever,
    "semantic": semantic_retriever
}

In [71]:
# Create separate testsets for each retrieval method
import copy
testset_retrievers = {
     "naive": copy.deepcopy(testset),
    "bm25": copy.deepcopy(testset),
    "compression": copy.deepcopy(testset),
    "multi_query": copy.deepcopy(testset),
    "parent_document": copy.deepcopy(testset),
    "ensemble": copy.deepcopy(testset),
    "semantic": copy.deepcopy(testset)
}

In [72]:
# factory function (returns objects) that return a new retrieval chain
def create_retriever_chain(retriever, retriever_name):
    """Create a RAG chain for a specific retriever"""
    return (
        {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
        | RunnablePassthrough.assign(context=itemgetter("context"))
        | {"answer": (rag_prompt | chat_model).with_config({"run_name": "llm"}) | StrOutputParser(), "contexts": itemgetter("context")}
    )

In [73]:
# Get context and responses
for name, testset in testset_retrievers.items():
  for test_row in testset:
    retriever = retrievers_to_evaluate.get(name)
    response = create_retriever_chain(retriever, name).invoke({"question" : test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["answer"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["contexts"]]

In [74]:
# Evaluate

from langsmith import Client
from datetime import datetime

# Add timestamp to make project names unique
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

client = Client()

dataset_name = f"Ragas Usecase + {timestamp}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description=f"Synthetic Data for evaluating retrieval pipelines + {timestamp}"
)

In [75]:
for test_row in testset_retrievers["naive"]:  # Use one testset as base
    client.create_example(
        inputs={"question": test_row.eval_sample.user_input},
        outputs={"ground_truth": test_row.eval_sample.reference},
        metadata={"context": test_row.eval_sample.reference_contexts},
        dataset_id=langsmith_dataset.id
    )

In order to use Ragas with LangChain, we first import all the metrics you want to use from ragas.metrics. Next we import the EvaluatorChain which is a langchain chain wrapper to convert a ragas metric into a langchain EvaluationChain.
> Note: We may encounter API connection errors or rate limiting during evaluation due to the high volume of requests being made to OpenAI's API.

In [76]:
import time
from datetime import datetime
import ragas
from ragas.integrations.langchain import EvaluatorChain
from ragas.metrics import (
    faithfulness, 
    context_precision, 
    context_recall
)
from langchain.smith import RunEvalConfig

# Add timestamp to make project names unique
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create the evaluation config
evaluation_config = RunEvalConfig(
    custom_evaluators=[
        EvaluatorChain(metric=faithfulness),
        EvaluatorChain(metric=context_precision),
        EvaluatorChain(metric=context_recall)
    ],
)

# Evaluate one retriever at a time
evaluation_results = {}

for name, retriever in retrievers_to_evaluate.items():
    print(f"🔍 Evaluating {name} retriever...")
    
    try:
        chain = create_retriever_chain(retriever, name)
        
        result = client.run_on_dataset(
            dataset_name=dataset_name,
            llm_or_chain_factory=chain,
            evaluation=RunEvalConfig(
                custom_evaluators=[
                    EvaluatorChain(faithfulness),
                    EvaluatorChain(context_precision),
                    EvaluatorChain(context_recall)
                ]
            ),
            project_name=f"{name}_{retriever}_{timestamp}"
        )
        
        evaluation_results[name] = result
        print(f"✅ {name} completed successfully")
        
    except Exception as e:
        print(f"❌ Error evaluating {name}: {e}")
        evaluation_results[name] = {"error": str(e)}
    
    # Add a small delay between evaluations
    import time
    time.sleep(2)

print("\n📊 All evaluations completed!")


🔍 Evaluating naive retriever...
View the evaluation results for project 'naive_tags=['Qdrant', 'OpenAIEmbeddings'] vectorstore=<langchain_community.vectorstores.qdrant.Qdrant object at 0x00000217726F2C10> search_kwargs={'k': 10}_20251009_223013' at:
https://smith.langchain.com/o/67d3d2c8-bb8c-4749-aa07-825356b10ae6/datasets/aabc9db6-61fd-4512-9b20-a8e283898f9c/compare?selectedSessions=a287c51e-6f4d-4141-b1da-6f0638a80aa1

View all tests for Dataset Ragas Usecase + 20251009_223008 at:
https://smith.langchain.com/o/67d3d2c8-bb8c-4749-aa07-825356b10ae6/datasets/aabc9db6-61fd-4512-9b20-a8e283898f9c
[>                                                 ] 0/10

Error evaluating run cddd9f64-8f71-46c7-9c71-9481b550db58 with EvaluatorChain
Traceback (most recent call last):
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\openai\_base_client.py", line 1529, in request
    response = await self._client.send(
               ^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1629, in send
    response = await self._send_handling_auth(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1657, in _send_handling_auth
    response = await self._send_handling_redirects(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1694, in _send_handling_redirects
    response = await self

[--------->                                        ] 2/10

Error evaluating run 4c77434c-7b0a-48a4-bc56-3b16fc0d154b with EvaluatorChain
Traceback (most recent call last):
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\openai\_base_client.py", line 1529, in request
    response = await self._client.send(
               ^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1629, in send
    response = await self._send_handling_auth(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1657, in _send_handling_auth
    response = await self._send_handling_redirects(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1694, in _send_handling_redirects
    response = await self

[----------------------------->                    ] 6/10

...

[---->                                             ] 1/10

Error evaluating run 4fec077c-4ce4-4093-b748-33517f6e4778 with EvaluatorChain
Traceback (most recent call last):
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\openai\_base_client.py", line 1529, in request
    response = await self._client.send(
               ^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1629, in send
    response = await self._send_handling_auth(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1657, in _send_handling_auth
    response = await self._send_handling_redirects(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1694, in _send_handling_redirects
    response = await self

[------------------->                              ] 4/10

Error evaluating run aa762552-8e1b-40b8-bea8-414736eb9468 with EvaluatorChain
Traceback (most recent call last):
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\openai\_base_client.py", line 1529, in request
    response = await self._client.send(
               ^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1629, in send
    response = await self._send_handling_auth(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<4 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1657, in _send_handling_auth
    response = await self._send_handling_redirects(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<3 lines>...
    )
    ^
  File "c:\Users\Inés\AIE2\09_Advanced_Retrieval\.venv\Lib\site-packages\httpx\_client.py", line 1694, in _send_handling_redirects
    response = await self

[------------------------------------------------->] 10/10
✅ semantic completed successfully

📊 All evaluations completed!


#### Analyisis

> NOTE: Small datasets make results unstable: each sample has outsized impact, plus retriever/LLM randomness, so scores and “best” models flip between runs. 

![Experiments](experiments/metrics_complete.png)

![Cost and Latency](experiments/cost_latency.png)

![Tokens](experiments/amount_tokens.png)

![Comparison](experiments/table.png)

## 📊 Retrieval Evaluation Results Analysis

### 🏆 Performance Rankings

Based on our evaluation using Ragas metrics (Context Precision, Context Recall, and Faithfulness), here are the key findings:

#### **Best Performers:**

1. **Contextual Compression (Rerank)** - Clear Winner
   - Context Precision: 0.98 (highest)
   - Context Recall: 1.00 (perfect)
   - Faithfulness: 0.72 (excellent)
   - Latency: 2.08s (moderate)
   - **Conclusion**: Best overall performance with balanced speed

2. **Parent Document Retriever** - Strong Second
   - Context Precision: 0.92 (excellent)
   - Context Recall: 0.96 (excellent)
   - Faithfulness: 0.65 (good)
   - Latency: 1.79s (fast)
   - **Conclusion**: Great balance of performance and speed

3. **Multi-Query Retriever** - Faithfulness Leader
   - Faithfulness: 0.76 (highest)
   - Context Recall: 0.93 (excellent)
   - Context Precision: 0.74 (decent)
   - Latency: 3.70s (slower)
   - **Conclusion**: Best when answer accuracy is critical

#### **Underperformers:**

- **Ensemble Retriever**: Surprisingly poor (Precision: 0.59, Faithfulness: 0.55) despite combining multiple methods. Likely due to noise from weaker retrievers like BM25.
- **BM25**: Lowest performance (Precision: 0.53, Recall: 0.65). Keyword-based matching struggles with semantic AI project queries.

### ⚡ Speed vs Quality Trade-offs

- **Fastest**: Naive (1.82s) and BM25 (1.86s) - but lower quality
- **Slowest**: Ensemble (5.35s) - with poor performance
- **Best Balance**: Compression (2.08s) with top quality

### 💰 Cost Analysis

All retrievers showed minimal cost differences (~$0.00-0.02 per run), with token usage being the main differentiator:
- **Highest tokens**: Multi-Query and Ensemble (~45K prompt tokens)
- **Lowest tokens**: BM25 and Compression (~11-12K prompt tokens)

### 🎯 Final Recommendation

**For this AI project dataset, use Contextual Compression (Rerank):**
- Achieves the best precision and perfect recall
- Reasonable latency and low token usage
- Significantly outperforms other methods on semantic queries

**Alternative**: Parent Document Retriever for scenarios requiring faster response times with still-excellent quality.

**Avoid**: Ensemble and BM25 for this type of semantic search task - they don't provide value for the added complexity or show poor performance on semantic queries.

### ⚠️ Limitations

As noted earlier, these results may not be fully representative due to:
1. Small dataset size (50 documents) leading to high variance between runs
2. Synthetic test data from Ragas rather than real user queries
3. Domain-specific data (AI/tech projects) that may not generalize to other domains

For production deployment, validate these findings with a larger dataset and real user feedback.
